# Fase 3 — Modelado | Actividad: 01 Baseline Naive (Limón Sutil y Dulce)

**Fase:** 3 — Modelado (v2 Reentrenamiento)
**Dominio:** Predicción de producción de limón (Sutil y Dulce), 2016–2025

---

## Objetivo

1. Establecer el **baseline trivial Naive** puro: `y_hat[t] = y[t-1]` (la producción del
   mes anterior como predicción para el mes actual). Sin variables exógenas, sin
   entrenamiento, deterministico.
2. Evaluar sobre las **mismas particiones** que usarán los modelos supervisados:
   **train 90 meses** (2016-07..2023-12), **val 12** (2024), **test 12** (2025).
3. Reportar **MAE, RMSE y R^2** por partición y por cultivo (Sutil y Dulce).
4. Calcular **Δs sobre TEST** usando la máscara de shock **ya validada** en la Fase 2
   (P75 de la variación mensual % de la producción real, en toneladas):
   **Sutil = 23.2%**, **Dulce = 33.9%**.
5. **Registrar el experimento** en el sistema de trazabilidad v2
   (`v2_reentrenamiento/experimentos/`).

## Definición de Δs (según `resultados/verificacion_delta_s_completa/verificar_delta_s.py` y
DECISIONES_METODOLOGICAS.md)

```
Δs = (MAE_shock − MAE_global) / MAE_global × 100      (solo sobre TEST)
```

- `MAE_shock` = MAE del modelo solo en los meses de TEST clasificados como shock.
- `MAE_global` = MAE del modelo sobre TODO el test (12 meses).
- Shock del mes `t` = `|100 × (y[t] − y[t−1]) / y[t−1]| > P75`.

> **Advertencia metodológica (v1):** la variación % mensual debe calcularse sobre la
> **serie real en toneladas**, NUNCA sobre variables escaladas/z-score (denominadores
> cercanos a 0 generan variaciones sintéticas de ±1000%). Por eso este notebook trabaja
> con `master_dataset_{sutil,dulce}_v2.csv` (escala física), no con la versión escalada.

## Entrada

- `v2_reentrenamiento/data/processed/master_dataset_sutil_v2.csv` (120×18)
- `v2_reentrenamiento/data/processed/master_dataset_dulce_v2.csv` (120×18)

## Salida (trazabilidad de experimentos)

- `v2_reentrenamiento/experimentos/exp_001_naive_sutil/{config.yaml, metricas.json, predicciones.csv}`
- `v2_reentrenamiento/experimentos/exp_001_naive_dulce/{config.yaml, metricas.json, predicciones.csv}`
- `v2_reentrenamiento/experimentos/REGISTRO_MAESTRO.csv` (una fila por experimento)

## Particiones (split ajustado, Fase 2)

| Partición | Rango | Meses | Shock en test (máscara P75) |
|---|---|---|---|
| Train | 2016-07 → 2023-12 | 90 | — |
| Val (2024) | 2024-01 → 2024-12 | 12 | — |
| Test (2025) | 2025-01 → 2025-12 | 12 | |var%| > 23.2% (Sutil) / 33.9% (Dulce) |


## 1. Configuración inicial


In [1]:
import os, json, warnings
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
pd.set_option('display.width', 240)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

while not os.path.exists('v2_reentrenamiento/data/processed'):
    os.chdir('..')
print('Raiz del proyecto:', os.getcwd())

# ---- Rutas de entrada/salida -------------------------------------------------
PROC     = 'v2_reentrenamiento/data/processed'
EXP_ROOT = 'v2_reentrenamiento/experimentos'

IN_SUTIL = f'{PROC}/master_dataset_sutil_v2.csv'
IN_DULCE = f'{PROC}/master_dataset_dulce_v2.csv'

# ---- Umbrales P75 YA VALIDADOS (Fase 2) sobre variación mensual % ------
# |100*(y_t - y_{t-1})/y_{t-1}| > umbral  =>  mes de shock
P75 = {'sutil': 23.2, 'dulce': 33.9}

# ---- Parametros del experimento ----------------------------------------
N_TRAIN, N_VAL, N_TEST = 90, 12, 12
FECHA = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')
print('P75 umbrales shock:', P75)
print('Particiones: train %s m | val %s m | test %s m' % (N_TRAIN, N_VAL, N_TEST))


Raiz del proyecto: C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-
P75 umbrales shock: {'sutil': 23.2, 'dulce': 33.9}
Particiones: train 90 m | val 12 m | test 12 m


## 2. Carga y preparación cronológica de la serie real


In [2]:
def cargar(path):
    df = pd.read_csv(path, encoding='utf-8-sig')
    df['fecha'] = df['año'].astype(str) + '-' + df['mes'].astype(str).str.zfill(2)
    df = df.sort_values(['año', 'mes']).reset_index(drop=True)
    return df

series = {'sutil': cargar(IN_SUTIL), 'dulce': cargar(IN_DULCE)}
for k, df in series.items():
    print(f'{k.upper()}: {df.shape} | rango {df["fecha"].iloc[0]} -> {df["fecha"].iloc[-1]} | '
          f'serie target: produccion_t_{k}')
    assert df.isna().sum().sum() == 0


SUTIL: (120, 19) | rango 2016-01 -> 2025-12 | serie target: produccion_t_sutil
DULCE: (120, 19) | rango 2016-01 -> 2025-12 | serie target: produccion_t_dulce


## 3. Particionado (train / val / test)


In [3]:
def particionar(df):
    train = df[(df['fecha'] >= '2016-07') & (df['fecha'] <= '2023-12')].reset_index(drop=True)
    val   = df[(df['fecha'] >= '2024-01') & (df['fecha'] <= '2024-12')].reset_index(drop=True)
    test  = df[(df['fecha'] >= '2025-01') & (df['fecha'] <= '2025-12')].reset_index(drop=True)
    assert len(train) == N_TRAIN, len(train)
    assert len(val)   == N_VAL,   len(val)
    assert len(test)  == N_TEST,  len(test)
    return train, val, test

for k in ('sutil', 'dulce'):
    serie = series[k]
    serie['particion'] = 'buffer'
    serie.loc[(serie['fecha'] >= '2016-07') & (serie['fecha'] <= '2023-12'), 'particion'] = 'train'
    serie.loc[serie['fecha'] >= '2024-01', 'particion'] = 'val'
    serie.loc[serie['fecha'] >= '2025-01', 'particion'] = 'test'
    tr, va, te = particionar(serie)
    print(f'{k.upper()}: train {len(tr)} ({tr.fecha.iloc[0]}..{tr.fecha.iloc[-1]}) | '
          f'val {len(va)} | test {len(te)} | buffer(pre-split) {int((serie.particion=="buffer").sum())}')
print()
print('Nota: los 6 meses 2016-01..2016-06 son buffer de lags (no se evaluan; solo aportan')
print('el valor previo para la primera prediccion de train 2016-07).')

COL_TARGET = {'sutil': 'produccion_t_sutil', 'dulce': 'produccion_t_dulce'}


SUTIL: train 90 (2016-07..2023-12) | val 12 | test 12 | buffer(pre-split) 6
DULCE: train 90 (2016-07..2023-12) | val 12 | test 12 | buffer(pre-split) 6

Nota: los 6 meses 2016-01..2016-06 son buffer de lags (no se evaluan; solo aportan
el valor previo para la primera prediccion de train 2016-07).


## 4. Modelo Naive: `y_hat[t] = y[t-1]`

No hay entrenamiento: la predicción del mes `t` es simplemente la producción real del mes
`t-1`. Para el primer mes de train (2016-07) la predicción usa el valor real 2016-06, que
sí existe en el dataset (la serie completa arranca en 2016-01). El naive **no consume
ninguna variable exógena**, por lo que su MAE/RMSE/R² da la cota de referencia que todo
modelo supervisado debe superar en los meses de comportamiento estable.


In [4]:
def metricas(real, pred):
    return {
        'mae':  float(mean_absolute_error(real, pred)),
        'rmse': float(np.sqrt(mean_squared_error(real, pred))),
        'r2':   float(r2_score(real, pred)),
    }

resumen = {}
for k in ('sutil', 'dulce'):
    df = series[k].copy()
    y   = df[COL_TARGET[k]]
    y_hat = y.shift(1)                      # y_hat[t] = y[t-1]
    df['predicho'] = y_hat
    df['error_abs'] = (y - y_hat).abs()
    df['error_cuad'] = (y - y_hat) ** 2

    resumen[k] = df
    print('=' * 66)
    print(f'NAIVE — {k.upper()} — métricas por partición (toneladas reales)')
    print('=' * 66)
    for part in ('train', 'val', 'test'):
        m = metricas(df.loc[df.particion == part, COL_TARGET[k]],
                     df.loc[df.particion == part, 'predicho'])
        print(f'  {part.upper():5s} | MAE = {m["mae"]:12,.2f} | RMSE = {m["rmse"]:12,.2f} | R² = {m["r2"]:8.4f}')


NAIVE — SUTIL — métricas por partición (toneladas reales)
  TRAIN | MAE =     3,356.88 | RMSE =     4,330.75 | R² =   0.7100
  VAL   | MAE =     3,078.13 | RMSE =     4,269.75 | R² =   0.7279
  TEST  | MAE =     4,704.29 | RMSE =     6,317.41 | R² =   0.4746
NAIVE — DULCE — métricas por partición (toneladas reales)
  TRAIN | MAE =        73.24 | RMSE =        86.17 | R² =   0.6429
  VAL   | MAE =        85.59 | RMSE =       100.85 | R² =   0.5739
  TEST  | MAE =        84.70 | RMSE =        97.71 | R² =   0.6686


## 5. Máscara de shock sobre TEST y Δs

La máscara P75 se aplica **solo sobre los meses de TEST**:

```
shock[t] = | 100 × (y[t] − y[t−1]) / y[t−1] |  >  P75
Δs  = (MAE_shock − MAE_global) / MAE_global × 100
```


In [5]:
print('=' * 66)
print('MÁSCARA DE SHOCK (P75) SOBRE TEST — 2025')
print('=' * 66)
delta_s = {}
for k in ('sutil', 'dulce'):
    df   = resumen[k].copy()
    df['var_pct'] = 100 * df[COL_TARGET[k]].pct_change()        # serie completa (2025-01 usa dic-2024)
    test = df[df.particion == 'test'].copy()
    test['shock'] = test['var_pct'].abs() > P75[k]

    mae_global = mean_absolute_error(test[COL_TARGET[k]], test['predicho'])
    shocks = test[test.shock]
    n_shock = len(shocks)
    if n_shock > 0:
        mae_shock = mean_absolute_error(shocks[COL_TARGET[k]], shocks['predicho'])
        ds = 100.0 * (mae_shock - mae_global) / mae_global
    else:
        mae_shock, ds = float('nan'), float('nan')

    delta_s[k] = {'n_shock': n_shock, 'mae_global': mae_global,
                  'mae_shock': mae_shock, 'delta_s_pct': ds}
    print(f'  {k.upper()}: umbral P75 = {P75[k]}% | shocks en test = {n_shock}')
    if n_shock:
        print('      meses shock:', list(shocks.fecha),
              '| var%:', [f'{v:+.1f}' for v in shocks.var_pct])
    print(f'      MAE_global(test) = {mae_global:,.2f} | MAE_shock = {mae_shock:,.2f} | '
          f'Δs = {ds:.2f}%' if n_shock else
          f'      MAE_global(test) = {mae_global:,.2f} | (sin shocks, Δs no definido)')

    # Persistir máscara en el dataframe resumen
    resumen[k]['shock'] = (100 * resumen[k][COL_TARGET[k]].pct_change().abs() > P75[k])        & (resumen[k].particion == 'test')


MÁSCARA DE SHOCK (P75) SOBRE TEST — 2025
  SUTIL: umbral P75 = 23.2% | shocks en test = 3
      meses shock: ['2025-01', '2025-07', '2025-11'] | var%: ['+44.3', '-32.1', '+57.1']
      MAE_global(test) = 4,704.29 | MAE_shock = 11,660.16 | Δs = 147.86%
  DULCE: umbral P75 = 33.9% | shocks en test = 3
      meses shock: ['2025-01', '2025-02', '2025-03'] | var%: ['+38.7', '+40.4', '+38.7']
      MAE_global(test) = 84.70 | MAE_shock = 138.26 | Δs = 63.23%


## 6. Registro del experimento en el sistema de trazabilidad v2

Por cada cultivo se crea la carpeta `exp_001_naive_<cultivo>/` con:

| Archivo | Contenido |
|---|---|
| `config.yaml` | Configuración reproducible del experimento (datos, split, umbral P75, Δs) |
| `metricas.json` | MAE/RMSE/R² por partición + Δs de test |
| `predicciones.csv` | Serie fecha, real, predicho, partición, shock |

Además se agrega una fila a `experimentos/REGISTRO_MAESTRO.csv` (columnas id, cultivo,
modelo, estado, datos, split, umbral, métricas resumen, Δs, fecha).


In [6]:
import yaml

EXPERIMENTOS = [
    ('sutil', 'exp_001_naive_sutil'),
    ('dulce', 'exp_001_naive_dulce'),
]

for cultivo, exp_id in EXPERIMENTOS:
    df      = resumen[cultivo]
    target  = COL_TARGET[cultivo]
    metrics = {p: metricas(df.loc[df.particion == p, target],
                           df.loc[df.particion == p, 'predicho'])
               for p in ('train', 'val', 'test')}
    d = delta_s[cultivo]
    metrics['test']['n_shock']     = d['n_shock']
    metrics['test']['mae_shock']   = d['mae_shock']
    metrics['test']['delta_s_pct'] = d['delta_s_pct']

    out_dir = os.path.join(EXP_ROOT, exp_id)
    os.makedirs(out_dir, exist_ok=True)

    # ---- config.yaml ----
    cfg = {
        'experimento': exp_id,
        'cultivo': cultivo.upper(),
        'modelo': 'Naive',
        'descripcion': 'Baseline trivial: y_hat[t] = y[t-1]. Sin variables exógenas.',
        'estado': 'ejecutado',
        'datos': {
            'entrada': f'master_dataset_{cultivo}_v2.csv',
            'fuente': 'Midagri (produccion_t)',
            'unidades': 'toneladas',
            'n_filas': int(len(df)),
            'rango': f'{df.fecha.iloc[0]}..{df.fecha.iloc[-1]}',
            'nulos': 0,
        },
        'split': {
            'train': {'rango': '2016-07..2023-12', 'meses': N_TRAIN},
            'val':   {'rango': '2024-01..2024-12', 'meses': N_VAL},
            'test':  {'rango': '2025-01..2025-12', 'meses': N_TEST},
        },
        'shock': {
            'definicion': '|100*(y_t - y_{t-1})/y_{t-1}| > P75',
            'escala': 'serie real en toneladas (NUNCA z-score)',
            'umbral_p75_pct': P75[cultivo],
            'sobre': 'TEST unicamente',
        },
        'delta_s': {
            'formula': '(MAE_shock - MAE_global) / MAE_global * 100',
            'observacion': 'deterioro relativo del modelo en meses de shock vs todo el test',
        },
        'entorno': {'python': '3.11', 'fecha_ejecucion': FECHA},
    }
    with open(os.path.join(out_dir, 'config.yaml'), 'w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False, default_flow_style=False)

    # ---- metricas.json ----
    with open(os.path.join(out_dir, 'metricas.json'), 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    # ---- predicciones.csv ----
    pred_out = pd.DataFrame({
        'fecha':      df['fecha'],
        'real_t':     df[target].round(2),
        'predicho_t': df['predicho'].round(2),
        'particion':  df['particion'],
        'shock_test': df['shock'].astype(bool),
    })
    pred_out.to_csv(os.path.join(out_dir, 'predicciones.csv'),
                    index=False, encoding='utf-8-sig')

    print(f'[OK] {exp_id}: config.yaml + metricas.json + predicciones.csv guardados')


[OK] exp_001_naive_sutil: config.yaml + metricas.json + predicciones.csv guardados
[OK] exp_001_naive_dulce: config.yaml + metricas.json + predicciones.csv guardados


## 7. Registro maestro de experimentos (`REGISTRO_MAESTRO.csv`)


In [7]:
REG = os.path.join(EXP_ROOT, 'REGISTRO_MAESTRO.csv')
COLUMNAS = ['experimento', 'cultivo', 'modelo', 'estado', 'datos', 'split',
            'umbral_p75', 'mae_train', 'mae_val', 'mae_test',
            'rmse_train', 'rmse_val', 'rmse_test',
            'r2_train', 'r2_val', 'r2_test',
            'n_shocks_test', 'mae_shock_test', 'delta_s_pct', 'fecha']

filas = []
for cultivo, exp_id in EXPERIMENTOS:
    df     = resumen[cultivo]
    m      = {p: metricas(df.loc[df.particion == p, COL_TARGET[cultivo]],
                          df.loc[df.particion == p, 'predicho']) for p in ('train','val','test')}
    d      = delta_s[cultivo]
    filas.append({
        'experimento':    exp_id,
        'cultivo':        cultivo.upper(),
        'modelo':         'Naive',
        'estado':         'ejecutado',
        'datos':          f'master_dataset_{cultivo}_v2.csv (120x18)',
        'split':          'train 2016-07..2023-12 (90) | val 2024 (12) | test 2025 (12)',
        'umbral_p75':     P75[cultivo],
        'mae_train':      round(m['train']['mae'], 2),
        'mae_val':        round(m['val']['mae'],   2),
        'mae_test':       round(m['test']['mae'],  2),
        'rmse_train':     round(m['train']['rmse'], 2),
        'rmse_val':       round(m['val']['rmse'],   2),
        'rmse_test':      round(m['test']['rmse'],  2),
        'r2_train':       round(m['train']['r2'], 4),
        'r2_val':         round(m['val']['r2'],     4),
        'r2_test':        round(m['test']['r2'],    4),
        'n_shocks_test':  d['n_shock'],
        'mae_shock_test': None if d['n_shock'] == 0 else round(d['mae_shock'], 2),
        'delta_s_pct':    None if d['n_shock'] == 0 else round(d['delta_s_pct'], 2),
        'fecha':          FECHA,
    })

df_reg = pd.DataFrame(filas, columns=COLUMNAS)
if os.path.exists(REG):
    prev = pd.read_csv(REG, encoding='utf-8-sig')
    df_reg = pd.concat([prev, df_reg], ignore_index=True)
# Idempotencia: una fila por experimento, se toma la ultima ejecucion
df_reg = df_reg.drop_duplicates(subset=['experimento'], keep='last')
os.makedirs(EXP_ROOT, exist_ok=True)
df_reg.to_csv(REG, index=False, encoding='utf-8-sig')
print('REGISTRO_MAESTRO.csv actualizado ->', len(df_reg), 'filas')
print('Ruta:', REG)


REGISTRO_MAESTRO.csv actualizado -> 2 filas
Ruta: v2_reentrenamiento/experimentos\REGISTRO_MAESTRO.csv


## 8. Resumen de resultados del baseline Naive


In [8]:
print('=' * 74)
print('RESULTADO BASELINE NAIVE (unidades reales: toneladas)')
print('=' * 74)
for cultivo, exp_id in EXPERIMENTOS:
    df    = resumen[cultivo]
    m     = {p: metricas(df.loc[df.particion == p, COL_TARGET[cultivo]],
                         df.loc[df.particion == p, 'predicho']) for p in ('train','val','test')}
    d     = delta_s[cultivo]
    print()
    print(f'-- {cultivo.upper()} ({exp_id}) --')
    print(f'  Train: MAE {m["train"]["mae"]:,.2f} | RMSE {m["train"]["rmse"]:,.2f} | R² {m["train"]["r2"]:+.4f}')
    print(f'  Val  : MAE {m["val"]["mae"]:,.2f} | RMSE {m["val"]["rmse"]:,.2f} | R² {m["val"]["r2"]:+.4f}')
    print(f'  Test : MAE {m["test"]["mae"]:,.2f} | RMSE {m["test"]["rmse"]:,.2f} | R² {m["test"]["r2"]:+.4f}')
    if d['n_shock']:
        print(f'  TEST shock (P75 {P75[cultivo]}%): {d["n_shock"]} meses | MAE_shock = {d["mae_shock"]:,.2f} | '
              f'Δs = {d["delta_s_pct"]:+.2f}%')
    else:
        print(f'  TEST: sin meses de shock según máscara P75 {P75[cultivo]}% (Δs no definido)')
print()
print('=== FIN ACTIVIDAD 01 — BASELINE NAIVE (v2) ===')
print('Siguiente paso: modelos SARIMA / Prophet contra esta cota de referencia.')


RESULTADO BASELINE NAIVE (unidades reales: toneladas)

-- SUTIL (exp_001_naive_sutil) --
  Train: MAE 3,356.88 | RMSE 4,330.75 | R² +0.7100
  Val  : MAE 3,078.13 | RMSE 4,269.75 | R² +0.7279
  Test : MAE 4,704.29 | RMSE 6,317.41 | R² +0.4746
  TEST shock (P75 23.2%): 3 meses | MAE_shock = 11,660.16 | Δs = +147.86%

-- DULCE (exp_001_naive_dulce) --
  Train: MAE 73.24 | RMSE 86.17 | R² +0.6429
  Val  : MAE 85.59 | RMSE 100.85 | R² +0.5739
  Test : MAE 84.70 | RMSE 97.71 | R² +0.6686
  TEST shock (P75 33.9%): 3 meses | MAE_shock = 138.26 | Δs = +63.23%

=== FIN ACTIVIDAD 01 — BASELINE NAIVE (v2) ===
Siguiente paso: modelos SARIMA / Prophet contra esta cota de referencia.
